In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
df = pd.read_csv("data/ChargingRecords.csv")

In [6]:
# fetching the duplicate records

duplicates = df[df.duplicated(keep=False)]

display(duplicates.head(30))

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
28856,329,214,1,public area,0,2022-02-12,7:22:40,2022-02-12,15:44:27,2022-02-12 7:22,2022-02-12 15:44,502,54.34
28857,329,214,1,public area,0,2022-02-12,7:22:40,2022-02-12,15:44:27,2022-02-12 7:22,2022-02-12 15:44,502,54.34
28858,329,214,1,public area,0,2022-02-15,19:29:42,2022-02-15,23:03:11,2022-02-15 19:29,2022-02-15 23:03,213,23.68
28859,329,214,1,public area,0,2022-02-15,19:29:42,2022-02-15,23:03:11,2022-02-15 19:29,2022-02-15 23:03,213,23.68
28861,329,214,1,public area,0,2022-02-17,18:53:55,2022-02-18,3:38:13,2022-02-17 18:53,2022-02-18 3:38,524,58.06
28862,329,214,1,public area,0,2022-02-17,18:53:55,2022-02-18,3:38:13,2022-02-17 18:53,2022-02-18 3:38,524,58.06
40669,461,371,1,apartment,0,2022-02-22,16:17:01,2022-02-22,21:38:44,2022-02-22 16:17,2022-02-22 21:38,322,37.74
40670,461,371,1,apartment,0,2022-02-22,16:17:01,2022-02-22,21:38:44,2022-02-22 16:17,2022-02-22 21:38,322,37.74
51498,276,640,0,company,1,2022-02-26,9:23:02,2022-02-26,10:03:07,2022-02-26 9:23,2022-02-26 10:03,40,15.04
51499,276,640,0,company,1,2022-02-26,9:23:02,2022-02-26,10:03:07,2022-02-26 9:23,2022-02-26 10:03,40,15.04


In [7]:
# it seems to be a exact duplicate records, so i drop those duplicate records

df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (72843, 13)


In [8]:
# verifying the duplicate records

print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [9]:
# Converting date columms - The original date/time columns are stored as text. Converting the combined date/time fields into proper datetime values.

for col in ["StartDatetime", "EndDatetime"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[["StartDatetime", "EndDatetime"]].dtypes)

StartDatetime    datetime64[ns]
EndDatetime      datetime64[ns]
dtype: object


In [10]:
print(df[["StartDatetime", "EndDatetime"]].isnull().sum())

StartDatetime    0
EndDatetime      0
dtype: int64


In [24]:
# Converting Date column

df["StartDay"] = pd.to_datetime(
    df["StartDay"], errors="coerce"
)

df["EndDay"] = pd.to_datetime(
    df["EndDay"], errors="coerce"
)

In [27]:
# Convert Time column

df["StartTime"] = pd.to_datetime(
    df["StartTime"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time

df["EndTime"] = pd.to_datetime(
    df["EndTime"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time

In [28]:
# Still StartTime and EndTime showas as object, When you use .dt.time, 
# Pandas converts the values into Python datetime.time objects, which are displayed as object
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72843 entries, 0 to 72855
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   UserID          72843 non-null  int64         
 1   ChargerID       72843 non-null  int64         
 2   ChargerCompany  72843 non-null  int64         
 3   Location        72843 non-null  object        
 4   ChargerType     72843 non-null  int64         
 5   StartDay        72843 non-null  datetime64[ns]
 6   StartTime       72843 non-null  object        
 7   EndDay          72843 non-null  datetime64[ns]
 8   EndTime         72843 non-null  object        
 9   StartDatetime   72843 non-null  datetime64[ns]
 10  EndDatetime     72843 non-null  datetime64[ns]
 11  Duration        72843 non-null  int64         
 12  Demand          72843 non-null  float64       
dtypes: datetime64[ns](4), float64(1), int64(5), object(3)
memory usage: 7.8+ MB


In [30]:
# to verify whether the conversion done for StartTime and EndTime?

print(type(df["StartTime"].iloc[0]))
print(type(df["EndTime"].iloc[0]))

<class 'datetime.time'>
<class 'datetime.time'>


In [35]:
# Investigate negative durations

negative_records = df[df["Duration"] < 0]

display(negative_records[[
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "CalculatedDuration",
    "Demand"
]])

,StartDatetime,EndDatetime,Duration,CalculatedDuration,Demand
55422,2022-02-15 15:04:00,2022-02-15 14:35:00,-29,-29.0,11.68
55425,2022-02-24 14:27:00,2022-02-24 14:18:00,-9,-9.0,34.56
55427,2022-01-17 14:47:00,2022-01-17 14:39:00,-8,-8.0,17.77
55455,2022-04-29 18:58:00,2022-04-29 18:52:00,-6,-6.0,31.90
55457,2022-05-07 16:24:00,2022-05-07 16:03:00,-21,-21.0,13.39
55458,2022-04-29 15:48:00,2022-04-29 15:43:00,-6,-5.0,19.36
55459,2022-03-27 15:17:00,2022-03-27 15:10:00,-7,-7.0,20.99
55460,2022-02-05 18:52:00,2022-02-05 18:50:00,-2,-2.0,31.59
55464,2022-05-08 12:02:00,2022-05-08 11:42:00,-20,-20.0,3.28
55465,2022-01-09 14:29:00,2022-01-09 14:20:00,-9,-9.0,31.68


In [36]:
# Check the relationship between the timestamps and duration

print("Negative original durations:",
      (df["Duration"] < 0).sum())

print("Negative calculated durations:",
      (df["CalculatedDuration"] < 0).sum())

print("Records with negative Duration but valid CalculatedDuration:")

display(
    df[
        (df["Duration"] < 0) &
        (df["CalculatedDuration"] >= 0)
    ][[
        "StartDatetime",
        "EndDatetime",
        "Duration",
        "CalculatedDuration"
    ]]
)

Negative original durations: 17
Negative calculated durations: 17
Records with negative Duration but valid CalculatedDuration:


,StartDatetime,EndDatetime,Duration,CalculatedDuration


In [34]:
negative_records = df[df["Duration"] < 0]

display(negative_records[[
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "CalculatedDuration",
    "Demand"
]])

,StartDatetime,EndDatetime,Duration,CalculatedDuration,Demand
55422,2022-02-15 15:04:00,2022-02-15 14:35:00,-29,-29.0,11.68
55425,2022-02-24 14:27:00,2022-02-24 14:18:00,-9,-9.0,34.56
55427,2022-01-17 14:47:00,2022-01-17 14:39:00,-8,-8.0,17.77
55455,2022-04-29 18:58:00,2022-04-29 18:52:00,-6,-6.0,31.90
55457,2022-05-07 16:24:00,2022-05-07 16:03:00,-21,-21.0,13.39
55458,2022-04-29 15:48:00,2022-04-29 15:43:00,-6,-5.0,19.36
55459,2022-03-27 15:17:00,2022-03-27 15:10:00,-7,-7.0,20.99
55460,2022-02-05 18:52:00,2022-02-05 18:50:00,-2,-2.0,31.59
55464,2022-05-08 12:02:00,2022-05-08 11:42:00,-20,-20.0,3.28
55465,2022-01-09 14:29:00,2022-01-09 14:20:00,-9,-9.0,31.68


In [38]:
# Removing invalid negative-duration records

df_backup = df.copy()

In [39]:
df = df[
    (df["Duration"] >= 0) &
    (df["CalculatedDuration"] >= 0)
].copy()

df.reset_index(drop=True, inplace=True)

In [40]:
# Ensuring the Negative-duration records are available

print("Remaining rows:", df.shape[0])

print("Negative Duration:",
      (df["Duration"] < 0).sum())

print("Negative CalculatedDuration:",
      (df["CalculatedDuration"] < 0).sum())

Remaining rows: 72826
Negative Duration: 0
Negative CalculatedDuration: 0


In [41]:
# Negative duration records removed

negative_records = df[df["Duration"] < 0]

display(negative_records[[
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "CalculatedDuration",
    "Demand"
]])

,StartDatetime,EndDatetime,Duration,CalculatedDuration,Demand


In [42]:
# Saving the cleaned Data set in to the folder

df.to_csv(
    "data/charging_records_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
